# Enterprise Financial Risk Intelligence & Fraud Forensics
## Notebook 01: Comprehensive Univariate Transaction Analysis & Distributional Physics

---

### Scientific Problem Formulation & Objectives:
Financial transaction ecosystems exhibit extreme asymmetric properties: **heavy-tailed transaction amounts**, **severe class imbalance** (less than 0.2% fraud), and **non-Gaussian PCA component distributions**. Detecting fraudulent behavior requires establishing rigorous baseline parametric and non-parametric statistical profiles for every feature before applying complex non-linear models.

### Mathematical Formulations:
1. **Fisher-Pearson Sample Skewness ($g_1$)**:
   $$g_1 = \frac{m_3}{m_2^{3/2}} = \frac{\frac{1}{n} \sum_{i=1}^n (x_i - \bar{x})^3}{\left[ \frac{1}{n} \sum_{i=1}^n (x_i - \bar{x})^2 \right]^{3/2}}$$
2. **Excess Kurtosis (Fisher Definition $\kappa$)**:
   $$\kappa = \frac{\frac{1}{n} \sum_{i=1}^n (x_i - \bar{x})^4}{\left[ \frac{1}{n} \sum_{i=1}^n (x_i - \bar{x})^2 \right]^2} - 3$$
3. **Power-Law Pareto Tail Decay**:
   $$P(X > x) \sim x^{-\alpha}, \quad x \ge x_{\min}$$
4. **Log-Transform Variance Stabilization**:
   $$y = \ln(1 + x), \quad \frac{\partial y}{\partial x} = \frac{1}{1 + x}$$

In [ ]:
from IPython.display import display
import os
import json
import warnings
import time
warnings.filterwarnings('ignore')

os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
sns.set_palette('deep')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

print("Financial Fraud Intelligence & Univariate Profiling environment initialized successfully.")

---
## 1. Raw Transaction Stream Ingestion & Schema Cataloging
Loading the master financial transaction repository ($N=284,807$) and verifying data integrity, null counts, and memory footprint.

In [ ]:
data_path = '../data/raw/creditcard.csv' if os.path.exists('../data/raw/creditcard.csv') else '../data/creditcard.csv'
if not os.path.exists(data_path):
    data_path = 'data/raw/creditcard.csv' if os.path.exists('data/raw/creditcard.csv') else 'creditcard.csv'

df = pd.read_csv(data_path)

total_rows, total_cols = df.shape
total_nulls = df.isnull().sum().sum()
mem_mb = df.memory_usage(deep=True).sum() / 1024**2

print(f"Total Transactions Recorded:    {total_rows:,}")
print(f"Total Feature Dimensions:       {total_cols}")
print(f"Total Missing / Null Values:    {total_nulls}")
print(f"Total Memory Consumption:       {mem_mb:.2f} MB")
print(f"Target Column (Class) Present:  {'Class' in df.columns}")

---
## 2. Comprehensive 31-Feature Parametric & Non-Parametric Profiling Matrix
Computing mean, standard deviation, median, interquartile range (IQR), skewness, excess kurtosis, and Jarque-Bera normality test metrics across all features.

In [ ]:
profile_records = []

for col in df.columns:
    series = df[col]
    mean_val = series.mean()
    std_val = series.std()
    median_val = series.median()
    p25 = series.quantile(0.25)
    p75 = series.quantile(0.75)
    iqr_val = p75 - p25
    min_val = series.min()
    max_val = series.max()
    skew_val = float(stats.skew(series))
    kurt_val = float(stats.kurtosis(series))
    
    jb_stat, jb_pval = stats.jarque_bera(series.iloc[:10000])
    
    profile_records.append({
        'Feature': col,
        'Mean': mean_val,
        'Std Dev': std_val,
        'Median': median_val,
        'IQR': iqr_val,
        'Min': min_val,
        'Max': max_val,
        'Skewness': skew_val,
        'Excess Kurtosis': kurt_val,
        'Normality Reject (p < 0.05)': jb_pval < 0.05
    })

profile_df = pd.DataFrame(profile_records)
display(profile_df)

---
## 3. Transaction Amount Forensic Physics: Heavy Tails & Power-Law Decay
Analyzing the empirical distribution of `Amount`:
1. Extreme disparity between median ($22.00) and maximum ($25,691.16).
2. Log-normal transformation vs. raw heavy-tail distribution.
3. Pareto power-law empirical survival curve:
   $$S(x) = P(X > x) \propto x^{-\alpha}$$

In [ ]:
amount_raw = df['Amount']
amount_log = np.log1p(amount_raw)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(amount_raw[amount_raw <= 500], bins=50, kde=True, color='#0284C7', ax=axes[0])
axes[0].set_title('Raw Transaction Amount (Clipped <= $500)', fontweight='bold')
axes[0].set_xlabel('Amount (USD / EUR)')
axes[0].set_ylabel('Transaction Count')

sns.histplot(amount_log, bins=50, kde=True, color='#10B981', ax=axes[1])
axes[1].set_title('Log-Transformed Amount: ln(1 + Amount)', fontweight='bold')
axes[1].set_xlabel('Log(Amount + 1)')
axes[1].set_ylabel('Density')

sorted_amounts = np.sort(amount_raw[amount_raw > 0])
ccdf = 1.0 - np.arange(len(sorted_amounts)) / float(len(sorted_amounts))
axes[2].loglog(sorted_amounts, ccdf, marker='.', linestyle='none', color='#EF4444', alpha=0.5)
axes[2].set_title('Empirical Pareto Survival Function P(Amount > x)', fontweight='bold')
axes[2].set_xlabel('Transaction Amount (Log Scale)')
axes[2].set_ylabel('Complementary Cumulative Probability (Log Scale)')

plt.tight_layout()
plt.show()
plt.close(fig)

print(f"Amount Mean:            ${amount_raw.mean():.2f}")
print(f"Amount Median:          ${amount_raw.median():.2f}")
print(f"Amount 99th Percentile: ${amount_raw.quantile(0.99):.2f}")
print(f"Amount Max:             ${amount_raw.max():.2f}")
print(f"Amount Skewness:        {stats.skew(amount_raw):.4f}")
print(f"Amount Excess Kurtosis: {stats.kurtosis(amount_raw):.4f}")

---
## 4. Temporal Dynamics & Cyclical Transaction Rates (`Time`)
`Time` records elapsed seconds from the first recorded transaction over a 48-hour observation window (172,792 seconds).
- Translating seconds into cyclic hour-of-day:
  $$\text{Hour} = \left( \frac{\text{Time}}{3600} \right) \pmod{24}$$
- Diurnal volume fluctuations (troughs during nighttime 02:00-06:00, peaks during daytime 10:00-18:00).

In [ ]:
df['Hour_of_Day'] = (df['Time'] / 3600.0) % 24
df['Day_Index'] = (df['Time'] / (3600.0 * 24.0)).astype(int) + 1

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df['Time'] / 3600.0, bins=48, color='#6366F1', kde=True, ax=axes[0])
axes[0].set_title('48-Hour Continuous Transaction Volume Timeline', fontweight='bold')
axes[0].set_xlabel('Elapsed Time (Hours from Inception)')
axes[0].set_ylabel('Transaction Frequency')

sns.histplot(data=df, x='Hour_of_Day', hue='Day_Index', bins=24, palette='Set2', kde=True, ax=axes[1])
axes[1].set_title('24-Hour Diurnal Cyclical Volume (Day 1 vs. Day 2)', fontweight='bold')
axes[1].set_xlabel('Hour of Day (0 to 23)')
axes[1].set_ylabel('Transaction Frequency')

plt.tight_layout()
plt.show()
plt.close(fig)

print(f"Time Elapsed Span: {df['Time'].max() / 3600.0:.2f} Hours ({df['Time'].max() / 86400.0:.2f} Days)")

---
## 5. Latent PCA Features ($V_1 \dots V_{28}$) Distributional Profiling
Visualizing the univariate KDE distributions of all 28 anonymized PCA components to assess symmetry, kurtosis, and outlier density.

In [ ]:
pca_cols = [f'V{i}' for i in range(1, 29)]
sample_pca = df[pca_cols].sample(min(50000, len(df)), random_state=42)

fig, axes = plt.subplots(7, 4, figsize=(18, 20))
axes = axes.flatten()

for i, col in enumerate(pca_cols):
    sns.kdeplot(sample_pca[col], color='#0284C7', fill=True, alpha=0.3, ax=axes[i])
    skew_val = stats.skew(df[col])
    kurt_val = stats.kurtosis(df[col])
    axes[i].set_title(f"{col} | Skew: {skew_val:.2f} | Kurt: {kurt_val:.1f}", fontsize=9, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

plt.suptitle('Univariate Probability Density Profiles for Latent Components V1 - V28', fontsize=14, fontweight='bold', y=1.002)
plt.tight_layout()
plt.show()
plt.close(fig)

---
## 6. Target Class Imbalance Geometry (`Class`)
Quantifying the empirical class distribution:
$$\text{Imbalance Ratio } \text{IR} = \frac{N_{\text{majority}}}{N_{\text{minority}}} = \frac{284,315}{492} \approx 577.87 : 1$$
Demonstrating that conventional accuracy ($99.83\%$) is completely degenerate.

In [ ]:
class_counts = df['Class'].value_counts()
class_pcts = df['Class'].value_counts(normalize=True) * 100

imbalance_ratio = class_counts[0] / class_counts[1]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(['Legitimate (Class 0)', 'Fraudulent (Class 1)'], class_counts.values, color=['#0284C7', '#EF4444'], width=0.5)
ax.set_yscale('log')
ax.set_title(f"Target Imbalance: {class_counts[0]:,} Legitimate vs. {class_counts[1]:,} Fraudulent (IR = {imbalance_ratio:.1f}:1)", fontweight='bold')
ax.set_ylabel('Transaction Count (Log Scale)')

for bar, count, pct in zip(bars, class_counts.values, class_pcts.values):
    height = bar.get_height()
    label_text = str(f"{count:,}") + "\n(" + str(f"{pct:.3f}%") + ")"
    ax.text(bar.get_x() + bar.get_width() / 2., height * 1.3, label_text, ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()
plt.close(fig)

print(f"Legitimate Transactions (Class 0): {class_counts[0]:,} ({class_pcts[0]:.4f}%)")
print(f"Fraudulent Transactions (Class 1): {class_counts[1]:,} ({class_pcts[1]:.4f}%)")
print(f"Exact Imbalance Ratio:             {imbalance_ratio:.2f} : 1")


---
## 7. Automated Univariate Profiling Manifest Export
Serializing statistical profiling artifacts to `data/univariate_profiling_manifest.json` for validation across downstream pipelines.

In [ ]:
manifest_dir = '../data' if os.path.exists('../data') else 'data'
os.makedirs(manifest_dir, exist_ok=True)

univariate_manifest = {
    "dataset_name": "Credit Card Transaction Fraud Dataset",
    "total_records": int(total_rows),
    "total_features": int(total_cols),
    "class_counts": {
        "legitimate_0": int(class_counts[0]),
        "fraudulent_1": int(class_counts[1])
    },
    "imbalance_ratio": float(imbalance_ratio),
    "amount_stats": {
        "mean": float(amount_raw.mean()),
        "median": float(amount_raw.median()),
        "std": float(amount_raw.std()),
        "max": float(amount_raw.max()),
        "skewness": float(stats.skew(amount_raw)),
        "kurtosis": float(stats.kurtosis(amount_raw))
    },
    "time_stats": {
        "min_seconds": float(df['Time'].min()),
        "max_seconds": float(df['Time'].max()),
        "duration_hours": float(df['Time'].max() / 3600.0)
    },
    "timestamp_generated": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
}

manifest_path = os.path.join(manifest_dir, 'univariate_profiling_manifest.json')
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(univariate_manifest, f, indent=2)

print(f"Univariate Profiling Manifest serialized to '{manifest_path}'")

---
## 8. Executive Univariate Risk & Statistical Scorecard

In [ ]:
univariate_scorecard = [
    {
        'Statistical Dimension': 'Transaction Amount Asymmetry',
        'Empirical Finding': f"Severe right-skewness (Skew = {stats.skew(amount_raw):.2f}, Kurtosis = {stats.kurtosis(amount_raw):.1f}) with 99% of transactions under $1,017, but maximum at ${amount_raw.max():,.2f}.",
        'Engineering Action': 'Must apply non-linear log1p or Yeo-Johnson power transformations in downstream feature engineering.'
    },
    {
        'Statistical Dimension': 'Extreme Class Imbalance',
        'Empirical Finding': f"Extreme 577.87:1 imbalance ratio (only 492 fraud cases out of 284,807 transactions; 0.1727% prevalence).",
        'Engineering Action': 'Standard accuracy metric is strictly banned; must optimize for PR-AUC, Cost-Utility, and asymmetric cost matrices.'
    },
    {
        'Statistical Dimension': 'Temporal Periodicity',
        'Empirical Finding': 'Distinct 24-hour diurnal cyclicity across the 48-hour recording window, with profound volume drops during nighttime.',
        'Engineering Action': 'Engineer cyclic sine/cosine hour transformations and rolling velocity acceleration windows.'
    },
    {
        'Statistical Dimension': 'Latent PCA Features (V1-V28)',
        'Empirical Finding': 'Features exhibit wide variability in tail thickness (kurtosis ranging from near-normal to > 100).',
        'Engineering Action': 'Robust scaling and outlier-resistant estimators are required for stability.'
    }
]

scorecard_df = pd.DataFrame(univariate_scorecard)
display(scorecard_df)

print(f"\n01_Comprehensive_Univariate_Transaction_Analysis.ipynb pipeline completed successfully.")